# الدرس 07: بناء نظام RAG محلي باستخدام مكتبة FAISS (الجزء الأول)
### Local Vector Search with Meta FAISS - Fundamentals & Indexing (Lecture 32)

في هذا الدرس نطبق الآتي:
- فهم واستخدام مكتبة Meta FAISS للبحث الدلالي السريع.
- مقارنة التضمينات وحساب تشابه جيب التمام (Cosine Similarity).
- بناء وحفظ واسترجاع الفهرس محلياً على القرص.
- البحث بالمسافة الإقليدية (L2 Distance) والفلترة بالبيانات الوصفية (Metadata Filtering).

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات
**آلية العمل:**
نقوم باستيراد مكتبة `FAISS` من لانج تشين، وتجهيز نموذج التضمين.
نوفر نموذج `HuggingFaceEmbeddings` كخيار محلي ومجاني 100% يعمل بدون إنترنت أو مفاتيح مدفوعة، مع إمكانية استخدام `OpenAIEmbeddings` تلقائياً إذا توفر مفتاح `OPENAI_API_KEY`.

In [ ]:
# import standard libraries and load environment variables
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database directory
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
faiss_db_dir = str(data_dir / "databases" / "faiss_index")

# import langchain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# initialize embedding model (local HuggingFace with OpenAI fallback)
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAIEmbeddings (text-embedding-3-small)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Using local HuggingFaceEmbeddings (all-MiniLM-L6-v2) for free offline execution")

print(f"Target FAISS Directory: {faiss_db_dir}")

### 2. تجهيز المستندات الأولية وإرفاق البيانات الوصفية (Metadata)
**آلية العمل:**
ننشئ قائمة من كائنات `Document` تمثل 4 مجالات أساسية: الذكاء الاصطناعي، تعلم الآلة، التعلم العميق، ومعالجة اللغات الطبيعية.
نرفق مع كل مستند حقول `metadata` مثل `topic` و `source` لتمكين التصفية الدقيقة لاحقاً.

In [ ]:
# create raw sample documents with metadata
sample_documents = [
    Document(
        page_content=(
            "Artificial Intelligence (AI) is the simulation of human intelligence by machines. "
            "It encompasses machine learning, deep learning, natural language processing, "
            "and computer vision to build systems capable of rational reasoning, problem solving, "
            "and autonomous decision making."
        ),
        metadata={"topic": "ai", "source": "intro_ai.txt"}
    ),
    Document(
        page_content=(
            "Machine Learning (ML) is a subset of AI where systems learn from historical data "
            "rather than relying on hardcoded procedural rules. Key paradigms include supervised learning "
            "(regression, classification), unsupervised learning (clustering), and reinforcement learning."
        ),
        metadata={"topic": "ml", "source": "ml_basics.txt"}
    ),
    Document(
        page_content=(
            "Deep Learning (DL) is a specialized branch of machine learning based on multi-layered artificial "
            "neural networks. Deep architectures such as Convolutional Neural Networks (CNNs) and Transformers "
            "excel at extracting hierarchical representations from complex visual and textual data."
        ),
        metadata={"topic": "dl", "source": "deep_learning.txt"}
    ),
    Document(
        page_content=(
            "Natural Language Processing (NLP) enables machines to understand, interpret, and generate human language. "
            "Core tasks include syntactic parsing, semantic analysis, sentiment detection, translation, and large language modeling."
        ),
        metadata={"topic": "nlp", "source": "nlp_overview.txt"}
    )
]

print(f"Created {len(sample_documents)} sample documents with metadata.")

### 3. تقطيع المستندات إلى Chunks متجانسة
**آلية العمل:**
نستخدم `RecursiveCharacterTextSplitter` بتقطيع النصوص إلى أحجام مناسبة (`chunk_size=500`, `chunk_overlap=50`).
تتوارث القطع الناتجة نفس البيانات الوصفية الأصلية تلقائياً.

In [ ]:
# initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

# split documents
chunks = text_splitter.split_documents(sample_documents)

print(f"Split {len(sample_documents)} documents into {len(chunks)} chunks:\n")
for idx, chunk in enumerate(chunks, start=1):
    print(f"Chunk [{idx}] | Topic: {chunk.metadata['topic']} | Chars: {len(chunk.page_content)}")

### 4. فحص التضمينات وتطبيق دالة حساب تشابه جيب التمام (Cosine Similarity)
**آلية العمل:**
1. نقارن بين `embed_query` (لتضمين سؤال واحد) و `embed_documents` (لتضمين دفعة نصوص).
2. نطبق دالة `compare_embeddings` بحساب الضرب النقطي ومقاييس الأطوال (Cosine Similarity).
3. نختبر التشابه الدلالي: نلاحظ أن (AI مقابل Artificial Intelligence) يعطي تشابهاً عالياً، بينما (AI مقابل Pizza) يعطي تشابهاً منخفضاً جداً.

In [ ]:
# helper function to compute cosine similarity between two texts
def compare_embeddings(text1: str, text2: str) -> float:
    v1 = np.array(embeddings.embed_query(text1))
    v2 = np.array(embeddings.embed_query(text2))
    cosine_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return float(cosine_sim)

# test semantic similarity comparisons
sim_synonym = compare_embeddings("AI", "Artificial Intelligence")
sim_unrelated = compare_embeddings("AI", "Italian Pizza")
sim_acronym = compare_embeddings("Machine Learning", "ML")

print(f"Similarity ('AI' vs 'Artificial Intelligence'): {sim_synonym:.4f} (High similarity)")
print(f"Similarity ('Machine Learning' vs 'ML'):       {sim_acronym:.4f} (High similarity)")
print(f"Similarity ('AI' vs 'Italian Pizza'):           {sim_unrelated:.4f} (Low similarity)")

### 5. إنشاء فهرس المتجهات بمكتبة FAISS (`FAISS.from_documents`)
**آلية العمل:**
نستخدم دالة `FAISS.from_documents` لتحويل كافة القطع إلى متجهات وحفظها داخل فهرس FAISS في الذاكرة الحية.
نفحص خاصية `vector_store.index.ntotal` للتأكد من عدد المتجهات التي تم إدراجها داخل الفهرس بنجاح.

In [ ]:
# build FAISS vector store from document chunks
vector_store = FAISS.from_documents(chunks, embeddings)

# check total indexed vectors
print(f"FAISS vector store created successfully!")
print(f"Total vectors in FAISS index (ntotal): {vector_store.index.ntotal}")

### 6. حفظ الفهرس على القرص وإعادة تحميله محلياً (`save_local` & `load_local`)
**آلية العمل:**
فهرس FAISS يعمل في الذاكرة (In-Memory). لجعله دائماً:
1. نستدعي `vector_store.save_local()` لحفظه في مجلد `data/databases/faiss_index`.
2. ينتج ملفين: `index.faiss` (المتجهات الثنائية) و `index.pkl` (نصوص المستندات والـ Metadata).
3. نعيد تحميل الفهرس عبر `FAISS.load_local()` مع تمرير `allow_dangerous_deserialization=True` لقراءة ملف الـ Pickle بأمان.

In [ ]:
# 1. save FAISS index locally to disk
vector_store.save_local(faiss_db_dir)
saved_files = [f.name for f in Path(faiss_db_dir).iterdir()]
print(f"FAISS index saved to: {faiss_db_dir}")
print(f"Generated files on disk: {saved_files}\n")

# 2. reload the index from disk
loaded_vector_store = FAISS.load_local(
    faiss_db_dir,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Successfully loaded FAISS index! Total vectors: {loaded_vector_store.index.ntotal}")

### 7. البحث الدلالي العادي (`similarity_search`)
**آلية العمل:**
نطرح سؤالاً دلالياً مثل *"What is deep learning?"* ونطلب أفضل 3 مستندات متطابقة (`k=3`).
تقوم FAISS بحساب التشابه واستخراج النصوص الأكثر صلة بالمفهوم.

In [ ]:
# perform standard similarity search
query = "What is deep learning and how does it relate to neural networks?"
results = loaded_vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
print(f"Top {len(results)} Similar Chunks:\n")
for idx, doc in enumerate(results, start=1):
    print(f"  [{idx}] (Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}):")
    print(f"      {doc.page_content[:95]}...\n")

### 8. البحث مع استخراج درجات المسافة الإقليدية (`similarity_search_with_score`)
**آلية العمل:**
تستخدم FAISS افتراضياً مقياس **L2 Euclidean Distance**.
- **المسافة الأقل تعني تطابقاً أعلى** (المسافة صفر تعني تطابق تام).
- نلاحظ أن المستند الأقرب يحصل على أقل رقم مسافة، والمستندات الأبعد تحظى بأرقام أكبر.

In [ ]:
# similarity search returning (document, L2_distance) pairs
results_with_scores = loaded_vector_store.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'\n")
print("Results with L2 Distance Scores (LOWER score = MORE similar):\n")
for idx, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"  Rank [{idx}] -> L2 Distance: {score:.4f}")
    print(f"    Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"    Content: {doc.page_content[:90]}...\n")

### 9. البحث مع التصفية بالبيانات الوصفية (Metadata Filtering)
**آلية العمل:**
في التطبيقات الواقعية، قد نرغب في البحث فقط داخل قسم معين (مثل: `topic = ml`).
نمرر معامل `filter={"topic": "ml"}`، فتقوم FAISS بحصر النتائج في المستندات التي تحقق هذا الشرط فقط واستبعاد أي مجال آخر.

In [ ]:
# similarity search with metadata filter
filter_dict = {"topic": "ml"}
filtered_results = loaded_vector_store.similarity_search(
    query="Explain key foundational concepts",
    k=3,
    filter=filter_dict
)

print(f"Search with Filter: {filter_dict}")
print(f"Returned {len(filtered_results)} matching chunks:\n")
for idx, doc in enumerate(filtered_results, start=1):
    print(f"  [{idx}] Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"      Content: {doc.page_content[:95]}...\n")

## 🎯 ملخص الدرس 07 (المحاضرة 32):
1. **Meta FAISS**: مكتبة فائقة السرعة طورتها Meta لمعالجة وتصنيف كميات ضخمة من المتجهات بكفاءة في الذاكرة الحية.
2. **حفظ مزدوج**: تنتج ملفين عند استدعاء `save_local`: ملف `index.faiss` للأرقام والمتجهات، وملف `index.pkl` للنصوص والبيانات الوصفية.
3. **فهم مقياس L2**: في FAISS، كلما قل رقم المسافة (`L2 Score`) كلما كانت النتيجة أقرب وأكثر دقة دلالياً.
4. **الخطوة التالية (المحاضرة 33)**: بناء خط أنابيب RAG متكامل باستخدام FAISS و LCEL ونموذج اللغة!